*© 2026 Paul Fergus. Free for student and research use — commercial use is strictly prohibited.*

# Lab 3 — Convolutional Neural Networks for image classification

**Module:** Deep Learning Concepts and Techniques (Computer Vision)  
**Week:** 3  
**Estimated time:** 150 minutes

---

## Learning outcomes

By the end of this lab you should be able to:

1. Explain what a 2D convolutional layer does, in terms of kernels, parameter sharing, and the dot-product calculation.
2. Describe the role of pooling and why it makes networks invariant to small translations.
3. Build a multi-block CNN in PyTorch using `nn.Conv2d`, `nn.MaxPool2d`, `nn.BatchNorm2d`, and `nn.Dropout`.
4. Split a dataset properly into **train / validation / test** rather than reusing the test set as a validation signal.
5. Train the CNN with mini-batch SGD/Adam, monitor validation loss, and select the best model with early stopping.
6. Apply the correct prediction-from-logits pattern for **multi-class** classification — using `argmax` over softmax — and explain why this differs from the binary case in Lab 2.
7. Save and reload a trained PyTorch model using `state_dict`, and predict on a single new image.
8. Visualise the learned filters of the first convolutional layer and reason about what they detect.

## Prerequisites

- **Lab 1** completed — you understand forward and backward propagation.
- **Lab 2** completed — you've built an MLP in PyTorch with a proper training loop, DataLoader, and early stopping.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 4 — Convolutional Neural Networks.
- Lecture 3: *Convolutions, kernels, and the receptive field*.

## The thread from last week

In Lab 2 you built an MLP for binary classification — 30 input features, fully-connected layers, one output. That works because each feature was a *separate measurement* (radius, texture, area, etc.) and the network had to learn how they relate to the target.

But what if your input is a **28×28 image**? An MLP would flatten it into 784 pixels and treat each one as an independent feature. The fact that pixel (12, 14) is spatially next to pixel (12, 15) — i.e., that nearby pixels are *related* — is invisible to the network. It has to relearn the structure of 2D space from scratch.

**Convolutional Neural Networks (CNNs) bake spatial structure into the architecture itself.** Instead of treating every pixel as independent, the network slides small *kernels* over the image, learning local patterns (edges, corners, textures) that compose into larger ones. This is the breakthrough that made deep learning dominate computer vision after 2012.

## Useful references

- [PyTorch tutorial: Training a classifier](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) — the canonical PyTorch CNN tutorial.
- [CS231n lecture notes on CNNs](https://cs231n.github.io/convolutional-networks/) — Stanford's gold-standard CNN reference.
- LeCun, Y., Bottou, L., Bengio, Y. and Haffner, P. (1998). *Gradient-based learning applied to document recognition*. Proceedings of the IEEE — the original LeNet paper.

---

## 1. What is a convolution, really?

A 2D convolution slides a small grid of weights — a **kernel** (or *filter*) — across an input image. At each position, the kernel multiplies the patch of pixels underneath it element-wise, sums the products, and writes that one number into the output. Then it slides one pixel along and does it again.

<img src="assets/convolution.svg" alt="3x3 kernel sliding over a 5x5 input, showing the dot product calculation" width="880"/>

The kernel above is a hand-designed **vertical edge detector**: a column of `+1`s on the right, a column of `-1`s on the left, zeros in the middle. When it sits over a vertical edge in the input (Step 1), the positive and negative responses *don't cancel* and the output is large. When it sits over a uniform region (Step 2), they cancel and the output is zero.

**Three ideas this captures, which together are the whole secret of CNNs:**

1. **Locality.** A kernel only looks at a small patch at a time. Real-world image features (edges, corners, textures) are local — you don't need to see the whole image to detect an edge.
2. **Parameter sharing.** The *same* 9 kernel weights are used at every position in the image. Compare this to an MLP, where every pixel-to-hidden-unit connection has its own weight. Parameter sharing is what makes CNNs efficient and translation-equivariant.
3. **Learnable kernels.** In a real CNN we don't hand-design the edge detector. We initialise the kernel weights randomly and let backpropagation figure out what patterns are useful. By the end of training, the first conv layer will have learned its own edge detectors, blob detectors, and texture detectors. We will *visualise* these at the end of the lab.

**Pooling.** After convolutions, we typically apply a **pooling** operation (usually `MaxPool2d(2)`) which downsamples by taking the maximum value in each 2×2 region. This makes the network slightly invariant to small translations (if the edge moves by 1 pixel, the max in that region is unchanged) and shrinks the spatial dimensions so deeper layers can have a larger effective receptive field.

## 2. The dataset

We will use **MNIST** — 70,000 small (28×28) grayscale images of handwritten digits from 0 to 9. It is the *fruit fly* of deep learning research: simple enough to train in minutes on a laptop, complex enough to teach the patterns that scale to real problems.

The data ships with `torchvision`, which downloads it automatically the first time you run this notebook. After that it's cached locally in the bind-mounted `data/` folder, so re-runs are instant.

> **Note on MNIST being 'solved'.** Top CNNs achieve over 99.7% accuracy on MNIST, which means it's saturated as a benchmark. We use it for teaching because the basics are clear, but real research has long moved on. Exercise 3 swaps MNIST for the harder Fashion-MNIST dataset, which is more representative of modern computer vision difficulty.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility — set every seed we can.
RNG_SEED = 7144
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# Use the GPU if available, otherwise fall back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

In [ ]:
# Standard preprocessing for MNIST: convert PIL image -> tensor (which scales to [0,1])
# and then normalise to roughly zero-mean, unit-variance. The mean and std below
# are the well-known MNIST training-set statistics.
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

transform = transforms.Compose([
    transforms.ToTensor(),                            # PIL -> tensor in [0, 1]
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

# Download once, then cached in data/. The path is relative so it works on every machine.
train_full = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

print(f"Training samples: {len(train_full)}")
print(f"Test samples: {len(test_set)}")
print(f"Image shape: {train_full[0][0].shape} (C, H, W)")
print(f"Classes: {train_full.classes}")

### 2.1 Train / validation / test split — the proper way

In Lab 2 I used the test set as a validation signal for early stopping, which I flagged as a simplification. Let's do it properly now.

MNIST gives us 60,000 training images and 10,000 test images. We'll carve **10,000 images off the training set** to use as our **validation set**, leaving 50,000 for actual training. The test set is held out and we don't touch it until the very end.

**The discipline matters.** If you tune your hyperparameters by checking test accuracy after every change, you're slowly leaking test information into your design choices. Your reported test number then becomes optimistic — and worse, you have no clean way to know how optimistic. Validation is what you check during development; test is what you check once, at the end.

In [ ]:
VAL_SIZE = 10_000
TRAIN_SIZE = len(train_full) - VAL_SIZE

# Seed the split so it's reproducible across runs.
generator = torch.Generator().manual_seed(RNG_SEED)
train_set, val_set = random_split(train_full, [TRAIN_SIZE, VAL_SIZE], generator=generator)

BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} ({TRAIN_SIZE} samples)")
print(f"Val batches:   {len(val_loader)} ({VAL_SIZE} samples)")
print(f"Test batches:  {len(test_loader)} ({len(test_set)} samples)")

### 2.2 What does the data actually look like?

Always look at your data. Below is one batch of 12 training images with their labels.

In [ ]:
# Grab one batch and visualise the first 12 images.
xb, yb = next(iter(train_loader))
print(f"Batch shape: {xb.shape}  (B, C, H, W)")
print(f"Pixel range after normalisation: {xb.min():.2f} to {xb.max():.2f}")

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for ax, img, lbl in zip(axes.flat, xb[:12], yb[:12]):
    # Un-normalise for display so the images look natural.
    display_img = img.squeeze().numpy() * MNIST_STD + MNIST_MEAN
    ax.imshow(display_img, cmap="gray")
    ax.set_title(f"label: {lbl.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Building the CNN

Our architecture is a small but representative CNN with **two convolutional blocks** followed by a small dense classifier:

| Layer                  | Output shape | What it does |
|------------------------|--------------|--------------|
| Input                  | (1, 28, 28)  | Grayscale image |
| `Conv2d(1→32, 3×3, padding=1)` + ReLU + BatchNorm | (32, 28, 28) | 32 different kernels detect low-level features |
| `MaxPool2d(2)`         | (32, 14, 14) | Downsample spatially |
| `Conv2d(32→64, 3×3, padding=1)` + ReLU + BatchNorm | (64, 14, 14) | 64 kernels combine the low-level features into mid-level ones |
| `MaxPool2d(2)`         | (64, 7, 7)   | Downsample again |
| `Flatten`              | (3136,)      | Turn the 3D feature map into a vector |
| `Linear(3136 → 128)` + ReLU + Dropout(0.3) | (128,) | Dense classifier head |
| `Linear(128 → 10)`     | (10,)        | One logit per class |

**Two design choices worth flagging:**

- `padding=1` on the 3×3 conv keeps the spatial size the same (`28→28`, `14→14`). Without padding you'd lose a pixel at each border per conv (`28→26→13→11→...`), making spatial reasoning annoying. **Same-padding is the default for almost all modern CNNs.**
- The final `Linear` outputs **logits**, *not* softmax probabilities. As in Lab 2, the loss function (`CrossEntropyLoss`) applies the softmax internally for numerical stability.

In [ ]:
class SmallCNN(nn.Module):
    """A two-block convolutional network for 28x28 grayscale image classification."""

    def __init__(self, n_classes: int = 10, dropout: float = 0.3):
        super().__init__()

        # Block 1: conv → batchnorm → relu → pool
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Classifier head
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        # Block 1: (B,1,28,28) -> (B,32,14,14)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # Block 2: (B,32,14,14) -> (B,64,7,7)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # Flatten -> (B, 3136)
        x = torch.flatten(x, start_dim=1)
        # Classifier
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)                          # logits — no softmax here


model = SmallCNN().to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

**A sanity check before training.** Always confirm that data of the right shape flows through your model and out the other side. This catches off-by-one shape errors in seconds rather than waiting for training to fail mysteriously.

In [ ]:
with torch.no_grad():
    sample_batch = xb[:4].to(DEVICE)
    logits = model(sample_batch)
print(f"Input batch shape:  {sample_batch.shape}")
print(f"Output logits shape: {logits.shape}  (expected: (4, 10))")

## 4. Training

We reuse the **`EarlyStopping` helper** and the **`run_epoch` pattern** from Lab 2. The only differences are:

1. The loss is `CrossEntropyLoss` (multi-class) instead of `BCEWithLogitsLoss` (binary).
2. We get accuracy from `argmax` over the 10 logits, not from thresholding a sigmoid.
3. We watch **validation** loss for early stopping — not test loss as I (deliberately) did last week.

This is now the canonical PyTorch training loop you'll be reusing for the rest of the module. Get comfortable with it.

In [ ]:
class EarlyStopping:
    """Stop training when a monitored metric has stopped improving."""

    def __init__(self, patience: int = 3, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.epochs_without_improvement = 0
        self.best_state = None
        self.should_stop = False

    def step(self, current_loss: float, model: nn.Module) -> None:
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.epochs_without_improvement = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                self.should_stop = True

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def run_epoch(model, loader, criterion, optimizer=None):
    """Run one epoch — training if optimizer is provided, evaluation otherwise."""
    is_train = optimizer is not None
    model.train(mode=is_train)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)

            logits = model(xb)
            loss = criterion(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)
            # For multi-class, the prediction is the argmax over the class dimension.
            preds = logits.argmax(dim=1)
            total_correct += (preds == yb).sum().item()
            total_samples += xb.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
# Re-initialise so this cell is re-runnable.
torch.manual_seed(RNG_SEED)
model = SmallCNN().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(patience=3)

MAX_EPOCHS = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

import time
start = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"epoch {epoch:>2d}  |  train loss {train_loss:.4f} acc {train_acc:.4f}  |  "
          f"val loss {val_loss:.4f} acc {val_acc:.4f}")

    early_stopping.step(val_loss, model)
    if early_stopping.should_stop:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

elapsed = time.perf_counter() - start
early_stopping.restore_best(model)
model = model.to(DEVICE)
print(f"\nTraining took {elapsed:.1f} seconds. Best val loss: {early_stopping.best_loss:.4f}")

In [ ]:
# Plot loss and accuracy curves for both splits.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["train_loss"], label="train", marker="o")
axes[0].plot(history["val_loss"], label="validation", marker="o")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy loss"); axes[0].set_title("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="train", marker="o")
axes[1].plot(history["val_acc"], label="validation", marker="o")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluating on the test set — and why argmax is right *this* time

In **Lab 2** we used a single sigmoid output for binary classification, and I flagged that the original lab incorrectly used `np.argmax` to convert it to a prediction. That was wrong because there was only one output column to pick from.

**In this lab the architecture is different.** We have **10 output units** (one logit per digit class), and `CrossEntropyLoss` interprets these as a softmax distribution over the 10 classes. The predicted class is the one with the highest logit — which is exactly what `argmax(dim=1)` returns. So this time, `argmax` is the right move.

The pattern is: **one output per class → argmax; one output total → threshold the sigmoid.** Always check which case you're in *before* writing the evaluation code.

In [ ]:
# Evaluate on the held-out test set. This is the FIRST time we touch it.
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(yb.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

test_acc = (y_pred == y_true).mean()
print(f"Test accuracy: {test_acc:.4f}\n")
print("Classification report:")
print(classification_report(y_true, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix — MNIST test set")
plt.tight_layout()
plt.show()

**What to observe.** The diagonal is bright — the network gets the vast majority of digits right. The off-diagonal cells tell you which digits get confused with which. Common confusions are pairs that look genuinely similar: 4↔9, 3↔5, 7↔1, etc. Have a look at the brightest off-diagonal cell in your run — does it match your intuition for which digits *look* similar?

## 6. Saving and loading the model

PyTorch's saving idiom is different from Keras's. Instead of saving the *whole model* (architecture + weights) into a file, PyTorch best practice is to save just the **`state_dict`** — a dictionary of all the model's parameter tensors — and recreate the architecture in code before loading the weights in.

**Why?** Because the architecture is *code*, and code is more reliable to version-control and inspect than a binary blob. You always know exactly what model you're loading weights into.

(Keras's `model.save('my_model.h5')` saved both, which is convenient but couples the model to the framework version. PyTorch leaves architecture in your hands and just saves the numbers.)

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

# Save weights
torch.save(model.state_dict(), "checkpoints/mnist_cnn.pt")
print("Saved to checkpoints/mnist_cnn.pt")

# Pretend we're a new Python session — recreate the architecture, then load weights.
fresh_model = SmallCNN().to(DEVICE)
fresh_model.load_state_dict(torch.load("checkpoints/mnist_cnn.pt", map_location=DEVICE, weights_only=True))
fresh_model.eval()

# Sanity-check the loaded model gives the same predictions as the trained one.
with torch.no_grad():
    sample_x, _ = next(iter(test_loader))
    sample_x = sample_x[:8].to(DEVICE)
    p1 = model(sample_x).argmax(dim=1)
    p2 = fresh_model(sample_x).argmax(dim=1)
print(f"Trained model predictions: {p1.cpu().numpy()}")
print(f"Loaded model predictions:  {p2.cpu().numpy()}")
print(f"Match: {torch.equal(p1, p2)}")

## 7. Predicting on a single image

When you want to predict on one image, you still need to feed the network a **4D tensor** of shape `(batch, channels, height, width)` — networks always expect a batch dimension, even if the batch is just one. We use `unsqueeze(0)` to add it.

In [ ]:
# Grab a single image from the test set.
single_x, single_y = test_set[0]
print(f"Image shape from dataset: {single_x.shape}  (C, H, W)")

# Add a batch dimension: (1, 1, 28, 28)
batched = single_x.unsqueeze(0).to(DEVICE)
print(f"After unsqueeze(0):       {batched.shape}  (B, C, H, W)")

with torch.no_grad():
    logits = model(batched)
    probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred = int(probs.argmax())

# Show the image and the predicted class.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
img_to_show = single_x.squeeze().numpy() * MNIST_STD + MNIST_MEAN
axes[0].imshow(img_to_show, cmap="gray")
axes[0].set_title(f"True: {single_y}  ·  Predicted: {pred}")
axes[0].axis("off")

axes[1].bar(range(10), probs, color="#0369a1")
axes[1].set_xticks(range(10))
axes[1].set_xlabel("class"); axes[1].set_ylabel("probability")
axes[1].set_title("Predicted class probabilities")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Visualising the learned filters

We talked about the kernel as a *hand-designed* edge detector in Section 1. But in a real CNN we don't design the kernels — we let backpropagation discover them. So what did our network actually learn?

The first convolutional layer has 32 filters, each of shape (1, 3, 3). We can pull them out and plot them as tiny 3×3 images. Lighter cells are higher weights, darker cells are lower (or negative).

**What to look for.** You'll see patterns that resemble simple visual features: a vertical line on one side, a horizontal line, a small blob, a diagonal. These are the building blocks the network discovered automatically for distinguishing digits — and they look surprisingly like the hand-designed filters used in classical computer vision before deep learning.

In [ ]:
# Pull the conv1 weights: shape (32, 1, 3, 3)
filters = model.conv1.weight.detach().cpu().squeeze(1).numpy()
print(f"Filter bank shape: {filters.shape}  (n_filters, height, width)")

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
vmin, vmax = filters.min(), filters.max()
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i], cmap="RdBu_r", vmin=vmin, vmax=vmax)
    ax.set_title(f"f{i}", fontsize=8)
    ax.axis("off")
plt.suptitle("Learned filters of conv1 (red = positive, blue = negative)", y=1.02)
plt.tight_layout()
plt.show()

---

## 9. Exercise 1 — architecture

Do **all three** of the following. For each, train the modified model on MNIST, record the **test accuracy**, and write a short reflection.

**(a) One conv block instead of two.** Modify `SmallCNN` to remove the second conv block entirely (just `conv1 → bn1 → relu → pool → flatten → fc1 → fc2`). Update the `Linear` input size accordingly. How does the test accuracy compare to the two-block model? Why might removing capacity hurt less than you'd expect on a dataset this simple?

**(b) Wider first layer.** Keep the two-block architecture but change `conv1` from 32 to 64 filters and `conv2` from 64 to 128. Test accuracy? Wall-clock training time? Is the trade-off worth it on MNIST?

**(c) 5×5 kernels vs 3×3.** Modify the network to use `kernel_size=5, padding=2` in both conv layers. How does this change the number of parameters? Does it change accuracy? Modern CNN design overwhelmingly favours stacks of small 3×3 kernels over single large kernels — given your results, why might that be?

In [ ]:
# Your code for Exercise 1 (a), (b), (c) here.


*Your written observations for Exercise 1:*

(a) 

(b) 

(c) 

## 10. Exercise 2 — what's the network actually doing?

Do **all three** of the following.

**(a) Examine the misclassified digits.** Find the indices in the test set where `y_pred != y_true`. Display 12 misclassified examples in a grid, with each one labelled by its true and predicted class. Are the mistakes 'reasonable' — i.e., would *you* find them ambiguous? Or do they look like things any human would get right?

**(b) Per-class accuracy.** Compute the test accuracy *separately for each of the 10 digits*. Which class does the model handle worst? Look back at the confusion matrix — does this match the worst row?

**(c) Feature map visualisation.** For one test image, plot the **output of `conv1`** (32 feature maps of size 28×28) as a grid. Pick a feature map that looks visually different from the others and explain in 1–2 sentences what pattern in the original digit it seems to be highlighting. *(Hint: you can grab intermediate activations by calling `model.conv1(image_batch)` directly — but remember to apply the normalisation transform first.)*

In [ ]:
# Your code for Exercise 2 (a), (b), (c) here.


*Your written observations for Exercise 2:*

(a) 

(b) 

(c) 

## 11. Exercise 3 — Fashion-MNIST: when the dataset gets harder

MNIST is too easy to be a meaningful benchmark. **Fashion-MNIST** is a drop-in replacement with the same shape (10 classes, 28×28 grayscale, 60k train + 10k test), but the images are clothing items photographed against a clean background:

| Label | Class       | Label | Class      |
|-------|-------------|-------|------------|
| 0     | T-shirt/top | 5     | Sandal     |
| 1     | Trouser     | 6     | Shirt      |
| 2     | Pullover    | 7     | Sneaker    |
| 3     | Dress       | 8     | Bag        |
| 4     | Coat        | 9     | Ankle boot |

It is much harder than MNIST. Top models reach ~93% on Fashion-MNIST compared to 99.7% on MNIST.

**Your task:**

**(a)** Load Fashion-MNIST exactly as we loaded MNIST above. Train the same `SmallCNN` architecture for 8 epochs with the same hyperparameters. Record the test accuracy.

**(b)** Plot a confusion matrix using the Fashion-MNIST class names (not just integer indices). Which class pairs get most confused? Are these confusions semantically reasonable (e.g. visually similar garments)?

**(c)** Write a short paragraph (4–6 sentences) explaining *why* Fashion-MNIST is harder than MNIST. Be specific: think about within-class variation, between-class similarity, and whether 28×28 is enough resolution for clothing photographs.

In [ ]:
# Loading Fashion-MNIST is identical to MNIST — just swap the dataset class.
# Use these mean/std for Fashion-MNIST (computed on its own training set):
FMNIST_MEAN, FMNIST_STD = 0.2860, 0.3530

# Your code for Exercise 3 here.


*Your written observations for Exercise 3:*

(a) 

(b) 

(c) 

---

## 12. Reflection questions

Answer in the markdown cells below. Aim for 2–4 sentences per question.

**Q1.** Explain in your own words what is meant by *parameter sharing* in a convolutional layer, and why it makes CNNs both more efficient and more *translation-equivariant* than a fully-connected network.

**Q2.** In Section 3 we used `padding=1` with 3×3 kernels to preserve the spatial size of the feature map. What would happen if we used `padding=0` with a stack of, say, ten 3×3 conv layers? Why might this be a problem on small inputs like 28×28?

**Q3.** A colleague claims: *"My MLP got 98% on MNIST. CNNs are overrated."* You suspect MNIST is just too easy to differentiate the architectures. How would you design a fair experiment to demonstrate when CNNs *actually* matter? Name **one** thing about the dataset you would change.

**Q4.** In Lab 2 we used `argmax` incorrectly (over a single-sigmoid output). In Lab 3 we used `argmax` correctly (over 10 logits). State the general rule in one sentence: when is `argmax` the right way to convert a model output into a prediction?

**Q5.** Looking at the conv1 filter visualisation in Section 8, the learned filters resemble simple visual primitives (edges, blobs). Suggest one practical implication of this for **transfer learning** — i.e., reusing parts of a pretrained network for a different task.

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

In **Lab 4** we will tackle a much harder image classification problem on **colour photographs**, and meet **data augmentation** — the technique that lets us multiply our effective training set by applying random rotations, flips, and crops on the fly. This is the key technique standing between toy datasets like MNIST and real-world computer vision.

Before leaving today, make sure:

- [ ] You have completed Exercises 1, 2, and 3 (including the Fashion-MNIST training run)
- [ ] You have answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)
- [ ] You have saved your work — the `labs/` folder is volume-mounted on your host